<a href="https://colab.research.google.com/github/francisdick/detecteur-langue-fon/blob/main/01_collecte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SEANCE 1 : FON AVEC ZENODO**

---



préparer le notebook et récupérer le fon

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Chemin de mon dossier dans "Mon Drive"
DOSSIER = "/content/drive/MyDrive/FRANCIS DICK CLAUDE IA/Projet NLP - Détecteur de Langue Fon/"

Mounted at /content/drive


Vérifie le chemin

In [ ]:
import os
os.listdir(DOSSIER)

Charge le corpus fon et explore-le

In [4]:
import pandas as pd

url_fon = "https://zenodo.org/records/4432712/files/Fon_French_Parallel_Data_25377.csv?download=1"
fon_brut = pd.read_csv(url_fon)

COL_FON = "Fon_Sentence"
COL_FR = "French_Translation"
print(fon_brut.shape)

(25377, 2)


chargement

In [4]:
reserve = fon_brut[[COL_FON]].rename(columns={COL_FON: "texte"})
reserve = reserve.dropna().drop_duplicates()
reserve = reserve[reserve["texte"].str.split().str.len() >= 2]

fon = reserve.sample(300, random_state=42)   # 42 = le même tirage qu'hier
fon["langue"] = "fon"
fon["source"] = "zenodo_4432712"
print(len(fon))   # 300

300


Nettoie et échantillonne 300 phrases

In [8]:
a_retirer = [24544, 13328, 7470, 10999, 11044, 4566, 17116, 5921, 17447, 13642,
             7212, 10479, 7370, 2521, 8520, 22027, 6366, 4394, 3842, 11133,
             16265, 21035, 5815, 25341, 17401, 3048, 6671, 8295, 18549, 2954,
             23525, 9641, 5591, 11499, 16641, 1880, 16075, 10496, 21811, 6415,
             18048, 24207, 16567, 9049, 3065, 4429, 2575]

# 1. Les phrases validées au premier tour
fon_final = fon.drop(a_retirer, errors="ignore")
print("Phrases validées :", len(fon_final))   # 260

# 2. Une file d'attente de remplaçantes, mélangée une fois pour toutes
file_attente = reserve.drop(fon.index).sample(frac=1, random_state=1)
file_attente = file_attente.drop(a_retirer, errors="ignore")

# 3. On prend juste ce qu'il faut, en tête de file
manque = 300 - len(fon_final)
complement = file_attente.head(manque).copy()
complement["langue"] = "fon"
complement["source"] = "zenodo_4432712"

fon_final = pd.concat([fon_final, complement])
print("Total :", len(fon_final))   # 300

Phrases validées : 260
Total : 300


Afficher le fon et le français côte à côte

In [9]:
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", None)

a_relire = complement.join(fon_brut[COL_FR])[["texte", COL_FR]]
a_relire.to_csv(DOSSIER + "a_relire.csv", index=True, encoding="utf-8")
a_relire

,texte,French_Translation
19947,alɔ gɔ tɔn,son aide
7474,a ɖi ɖɔ e na lɛ kɔ wa,penses-tu qu'il va revenir
9918,hùn cé kɛ̀,mon coeur tressaille de peur
9891,agban ɔ kpɛ́n dín,la charge est très lourde
11064,kplɔn nukpinkplɔn tɔn lε,apprend ses leçons
10254,a nɔ lɛn blewun,tu calcules vite
18041,e ɖò hɔnlihɛ́nnú,il est sur le seuil
2340,e j'àyĭ ɖò kútɔ́nú,il a atterri à cotonou
1986,amɔ nukplɔnmεtɔ wε nyi,mais c'est un professeur
7259,nu ɖokpo ɔ wɛ nyi nɔvi ce kpodo dada ce kpo,mon frère et ma sœur sont pareils


vérification avant enregistrement du fichier

In [11]:
print("Nombre de phrases :", len(fon_final))                          # 300
print("Doublons de texte :", fon_final["texte"].duplicated().sum())   # 0
print("Phrases rejetées encore présentes :", fon_final.index.isin(a_retirer).sum())  # 0

Nombre de phrases : 300
Doublons de texte : 0
Phrases rejetées encore présentes : 0


Pour lire plus confortablement : un Google Sheet

In [12]:
fon_final.to_csv(DOSSIER + "fon_zenodo_300.csv", index=False, encoding="utf-8")
print("Fichier enregistré ✅")

Fichier enregistré ✅


# **SEANCE 2 : FRANCAIS ET ANGLAIS AVEC TATOEBA**

---



Cellule A : vérifier le nom exact des fichiers

In [13]:
[f for f in os.listdir(DOSSIER) if f.endswith(".bz2")]

['fra_sentences.tsv.bz2', 'eng_sentences.tsv.bz2']

Cellule B : charger les deux corpus

In [14]:
def charger_tatoeba(nom_fichier):
    df = pd.read_csv(DOSSIER + nom_fichier, sep="\t", header=None,
                     names=["id", "code", "texte"], quoting=3)
    return df.set_index("id")   # le numéro Tatoeba devient l'identifiant de chaque phrase

fr_brut = charger_tatoeba("fra_sentences.tsv.bz2")
en_brut = charger_tatoeba("eng_sentences.tsv.bz2")
print("Français :", len(fr_brut), "| Anglais :", len(en_brut))
fr_brut.head()

Français : 730079 | Anglais : 2040567


,code,texte
id,,
1115,fra,"Lorsqu'il a demandé qui avait cassé la fenêtre, tous les garçons ont pris un air innocent."
1279,fra,Je ne supporte pas ce type.
1441,fra,Pour une fois dans ma vie je fais un bon geste... Et ça ne sert à rien.
1593,fra,Ne tenez aucun compte de ce qu'il dit.
3091,fra,Essayons quelque chose !


Cellule C : une fonction de sélection réutilisable

In [18]:
PRENOMS = r"\b(?:Sami|Layla|Ziri|Rima|Yanni|Mennad|Skura|Jugurtha|Youba)\b"

def selectionner(df, langue, a_retirer, n=300):
    nb_mots = df["texte"].str.split().str.len()
    propre = df[(nb_mots >= 2) & (nb_mots <= 25)]
    propre = propre[~propre["texte"].duplicated()]

    # 1. On mélange TOUJOURS le même ensemble : l'ordre ne change jamais
    file_attente = propre.sample(frac=1, random_state=42)

    # 2. Ensuite seulement, on filtre (prénoms, duo Tom + Mary, rejets)
    t = file_attente["texte"]
    duo = t.str.contains(r"\bTom\b") & t.str.contains(r"\b(?:Mary|Marie)\b")
    file_attente = file_attente[~t.str.contains(PRENOMS) & ~duo]
    file_attente = file_attente.drop(a_retirer, errors="ignore")

    choix = file_attente.head(n)[["texte"]].copy()
    choix["langue"] = langue
    choix["source"] = "tatoeba_" + choix.index.astype(str)
    return choix

Cellule D : tirer les phrases et créer les fichiers de relecture

In [20]:
a_retirer_fr = [5375124, 13804370, 13484157, 12657569, 4471770, 14055666, 12676999,
                7694199, 3879718, 3249634, 13538912, 11894968, 12871490, 11049390,
                11803409, 11035761, 8706554, 11351854, 13886364, 13482820, 13563910,
                13483561, 11682127, 5278890, 10791490, 5606067, 8944708, 12754041,
                7060866, 2922321, 450592, 3271491, 7060866, 2922321, 450592, 3271491,
                12103373, 13096775]

a_retirer_en = [6437472, 6148993, 6326445, 11207759, 7288605, 10052411, 11670829,
                12503884, 6273411, 6480070, 10171567, 11043368, 2723174, 11227587,
                11573996, 11504038, 9953916, 6521379, 6649998, 10236810, 266896,
                10134149, 6698213, 8057880, 10171564, 11415781, 12043420, 7161158,
                6529250, 11219222, 12649234, 7288793, 8015739, 10210475, 7199697,
                12563747, 6953167, 13166087, 6873962, 8232890, 7992128, 6715409,
                6490264, 6522963, 6518865, 6416327, 10774066, 10165489, 6354994,
                6474296, 7165152, 8159275, 6417383, 12050516, 6097607, 7147007,
                8616153, 6416079, 10573652, 7985546, 7202623, 6527327, 8616153, 6416079, 10573652, 7985546, 7202623, 6527327,
                11793862, 9235096]

fr = selectionner(fr_brut, "fr", a_retirer_fr)
en = selectionner(en_brut, "en", a_retirer_en)
print("Français :", len(fr), "| Anglais :", len(en))   # 300 | 300

fr[["texte"]].to_csv(DOSSIER + "a_relire_fr.csv", encoding="utf-8")
en[["texte"]].to_csv(DOSSIER + "a_relire_en.csv", encoding="utf-8")
print("Fichiers de relecture créés ✅")

Français : 300 | Anglais : 300
Fichiers de relecture créés ✅


In [21]:
print("FR :"); print(fr.tail(2)["texte"])
print("EN :"); print(en.tail(2)["texte"])

FR :
id
12583055    Il ne s'est pas intégré dans votre groupe.
10303482                Ce sont toujours des ennemies.
Name: texte, dtype: object
EN :
id
847151    Neither of them cares for strenuous sports.
18875                              Turn on your back.
Name: texte, dtype: object


In [22]:
# Contrôle final
for nom, df, rej in (("FR", fr, a_retirer_fr), ("EN", en, a_retirer_en)):
    print(nom, "| phrases :", len(df),                                   # 300
          "| doublons :", df["texte"].duplicated().sum(),                # 0
          "| rejetées présentes :", df.index.isin(rej).sum())            # 0

# Enregistrement
fr.to_csv(DOSSIER + "tatoeba_fr_300.csv", index=False, encoding="utf-8")
en.to_csv(DOSSIER + "tatoeba_en_300.csv", index=False, encoding="utf-8")
print("Fichiers enregistrés ✅")

FR | phrases : 300 | doublons : 0 | rejetées présentes : 0
EN | phrases : 300 | doublons : 0 | rejetées présentes : 0
Fichiers enregistrés ✅


# **SEANCE 3 : ASSEMBLAGE**

---



Cellule E : charger les trois fichiers

In [5]:
import unicodedata

fon_ok = pd.read_csv(DOSSIER + "fon_zenodo_300.csv")
fr_ok  = pd.read_csv(DOSSIER + "tatoeba_fr_300.csv")
en_ok  = pd.read_csv(DOSSIER + "tatoeba_en_300.csv")

for nom, df in (("fon", fon_ok), ("fr", fr_ok), ("en", en_ok)):
    print(nom, df.shape, df.columns.tolist())   # (300, 3) pour chacun

fon (300, 3) ['texte', 'langue', 'source']
fr (300, 3) ['texte', 'langue', 'source']
en (300, 3) ['texte', 'langue', 'source']


Cellule F : corriger les caractères imposteurs

In [6]:
IMPOSTEURS = {"ε": "ɛ",   # epsilon grec → ɛ du fon
              "ð": "ɖ"}   # eth islandais → ɖ du fon

def normaliser(texte):
    for faux, vrai in IMPOSTEURS.items():
        texte = texte.replace(faux, vrai)
    return unicodedata.normalize("NFC", texte)

print("Phrases à corriger :", fon_ok["texte"].str.contains("ε|ð").sum())   # 35
fon_ok["texte"] = fon_ok["texte"].apply(normaliser)
print("Après correction :", fon_ok["texte"].str.contains("ε|ð").sum())     # 0

Phrases à corriger : 35
Après correction : 0


Cellule G : créer les versions « WhatsApp »

In [7]:
SIMPLES = {"ɔ": "o", "ɛ": "e", "ɖ": "d", "Ɔ": "O", "Ɛ": "E", "Ɖ": "D"}

def version_whatsapp(texte):
    for special, simple in SIMPLES.items():
        texte = texte.replace(special, simple)
    texte = unicodedata.normalize("NFD", texte)    # sépare chaque lettre de son ton
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")  # retire les tons
    return unicodedata.normalize("NFC", texte)

choisies = fon_ok.sample(150, random_state=7).index
fon_ok.loc[choisies, "texte"] = fon_ok.loc[choisies, "texte"].apply(version_whatsapp)
fon_ok.loc[choisies, "source"] = "zenodo_4432712_simplifie"

print("Avec ɔ, ɛ ou ɖ :", fon_ok["texte"].str.contains("[ɔɛɖ]").sum())   # 130
print("Doublons :", fon_ok["texte"].duplicated().sum())                   # 0
fon_ok.loc[choisies].head()

Avec ɔ, ɛ ou ɖ : 130
Doublons : 0


,texte,langue,source
254,ku d'ewu,fon,zenodo_4432712_simplifie
57,nu ahen o,fon,zenodo_4432712_simplifie
150,dada xo maga gbowun de b'e vivi kaka,fon,zenodo_4432712_simplifie
66,adu ton bi wo,fon,zenodo_4432712_simplifie
114,e do tavo o glo,fon,zenodo_4432712_simplifie


Cellule H : assembler phrases.csv

In [8]:
phrases = pd.concat([fon_ok, fr_ok, en_ok], ignore_index=True)
phrases = phrases.sample(frac=1, random_state=42).reset_index(drop=True)   # on mélange les langues

print(phrases["langue"].value_counts())                       # 300 / 300 / 300
print("Doublons :", phrases["texte"].duplicated().sum())      # 0
print("Cases vides :", phrases.isna().sum().sum())            # 0

phrases.to_csv(DOSSIER + "phrases.csv", index=False, encoding="utf-8")
print("phrases.csv enregistré ✅")

langue
fon    300
en     300
fr     300
Name: count, dtype: int64
Doublons : 0
Cases vides : 0
phrases.csv enregistré ✅


Cellule I : créer pieges.csv

In [9]:
pieges = pd.DataFrame([
    # Fon contenant des mots français (rejetées en séance 1)
    ("un ɖɔ nu we ɖɔ hwɛ nɛ zan 'ordinateur' ɔ", "fon", "mot français dans du fon"),
    ("mi ɖo 'manivelle' ɔ lilɛ wɛ",              "fon", "mot français dans du fon"),
    ("mi xɔ menthe gba ɖokpo",                   "fon", "mot français dans du fon"),
    # Fon contenant des prénoms ou des lieux
    ("brenda ɔ mɛsi glɛnsingbe kplɔnmɛtɔ wɛ",    "fon", "prénom"),
    ("patrick wɛ nɛ a fɔn gbɔn",                 "fon", "prénom"),
    ("un ɖo akassato",                           "fon", "lieu, phrase courte"),
    # Français et anglais avec lieux inconnus ou mélange
    ("Vous déménagez à Igoufaf.",                "fr",  "lieu inconnu"),
    ("Reading Tatoeba sentences, I sometimes have déjà vu.", "en", "mot français dans de l'anglais"),
    ("Le meeting est cancel",                    "fr",  "mots anglais dans du français"),
    # Très court ou sans langue
    ("Ok",                                       "inconnu", "un seul mot"),
    ("Merci",                                    "fr",  "un seul mot"),
    ("Bohicon",                                  "inconnu", "nom propre seul"),
    # Mélanges fon-français façon WhatsApp (fon tiré du corpus validé)
    ("Bonjour patron, a fɔn ganji à ?",          "fon",   "salutation française + fon"),
    ("a fon ganji a ?",                          "fon",   "fon sans lettres spéciales, très court"),
    ("Désolé, un ma se nɛ́",                     "fon",   "excuse française + fon"),
    ("Aujourd'hui nù cí mì vraiment",            "mixte", "fon encadré de français"),
    ("Merci beaucoup, mi kú dó mɛkanxuxo",       "mixte", "remerciement dans les deux langues"),
    ("e ɖo réunion ɔ mɛ",                        "fon",   "mot français dans du fon"),
    ("un na wá demain dandàn",                   "fon",   "mot français dans du fon"),
    ("ɛɛn, j'arrive dans 5 minutes",             "fr",    "fon court + français"),
    ("un ɖò kwɛ́ yɔywɛ ă, tu as la monnaie ?",   "mixte", "moitié fon, moitié français"),
    ("Ok, mì nà yí bó wá",                       "fon",   "mot anglais + fon"),
], columns=["texte", "langue_attendue", "type"])

pieges.to_csv(DOSSIER + "pieges.csv", index=False, encoding="utf-8")
print(len(pieges), "phrases pièges enregistrées ✅")   # 22

22 phrases pièges enregistrées ✅


Cellule J : rédiger SOURCES.md

In [10]:
sources = """# Sources des données

## Fon — Fon French Daily Dialogues Parallel Data
- Auteurs : Bonaventure Dossou, Fabroni Yoclounon, Ricardo Ahounvlamè, Chris Emezue (2021)
- Lien : https://doi.org/10.5281/zenodo.4432712
- Licence : CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/
- Utilisation : 300 phrases relues et validées à la main.
- Modifications : correction de caractères (ε → ɛ, ð → ɖ), normalisation Unicode NFC,
  150 phrases converties sans tons ni lettres spéciales (source « zenodo_4432712_simplifie »).
- Articles à citer :
  - Dossou & Emezue (2021), *Crowdsourced Phrase-Based Tokenization for Low-Resourced
    Neural Machine Translation: The Case of Fon Language*, arXiv:2103.08052
  - Dossou & Emezue (2020), *FFR v1.1: Fon-French Neural Machine Translation*, WiNLP 2020

## Français et anglais — Tatoeba
- Lien : https://tatoeba.org
- Licence : CC BY 2.0 FR — https://creativecommons.org/licenses/by/2.0/fr/
- Utilisation : 300 phrases par langue, relues et validées.
- Attribution : la colonne `source` contient le numéro de chaque phrase (tatoeba_<id>).
  Page de la phrase et de son auteur : https://tatoeba.org/fr/sentences/show/<id>

## Phrases pièges (pieges.csv)
- Phrases issues des corpus ci-dessus, et phrases mêlant fon et français composées
  à partir du corpus Zenodo, puis validées par Francis Dick ACCROMBESSY.
"""
with open(DOSSIER + "SOURCES.md", "w", encoding="utf-8") as f:
    f.write(sources)
print("SOURCES.md enregistré ✅")

SOURCES.md enregistré ✅
